# M2.Ex2: Vision Language Models (VLMs)

Two tasks:
1. **Information Extraction** with Docling + GraniteDocling — pull structured data out of receipts / PDFs / Word docs.
2. **Arabic Document Processing** with Qari-OCR — OCR Arabic images and a folder of Arabic PDFs.


---
## Task 1: Information Extraction with a VLM

We use Docling's `DocumentExtractor`, which runs a VLM (GraniteDocling) under the hood
and returns structured JSON matching a schema we define with Pydantic.


In [ ]:
# Install Docling with VLM extras (one-time, ~a few minutes on Colab)
%pip install -q "docling[vlm]" pydantic rich


In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field
from rich import print

from docling.datamodel.base_models import InputFormat
from docling.document_extractor import DocumentExtractor

# Allow images and PDFs as input
extractor = DocumentExtractor(allowed_formats=[InputFormat.IMAGE, InputFormat.PDF])


### 1a. Extract from a receipt image

We define the fields we want as a Pydantic model. The `examples=` hints help the VLM
understand what each field should look like.


In [ ]:
class LineItem(BaseModel):
    name: str = Field(examples=["Cappuccino", "Cheeseburger"])
    quantity: int = Field(default=1, examples=[1, 2])
    price: float = Field(examples=[12.50, 4.99])


class Receipt(BaseModel):
    merchant: Optional[str] = Field(default=None, examples=["Starbucks", "McDonald's"])
    date: Optional[str] = Field(default=None, examples=["2024-09-15"])
    items: List[LineItem] = Field(default_factory=list)
    subtotal: Optional[float] = Field(default=None, examples=[42.00])
    tax: Optional[float] = Field(default=None, examples=[3.36])
    total: Optional[float] = Field(default=None, examples=[45.36])
    currency: Optional[str] = Field(default=None, examples=["USD", "SAR"])


In [ ]:
# A public sample receipt image.
receipt_url = "https://templates.invoicehome.com/receipt-template-us-neat-750px.png"

result = extractor.extract(source=receipt_url, template=Receipt)
print(result.pages)


In [ ]:
# Validate the raw dict back into a Pydantic object so we can use it like normal data.
receipt = Receipt.model_validate(result.pages[0].extracted_data)

print(f"Merchant: {receipt.merchant}")
print(f"Date:     {receipt.date}")
print(f"Total:    {receipt.total} {receipt.currency or ''}")
print("Items:")
for item in receipt.items:
    print(f"  - {item.quantity} x {item.name}  @ {item.price}")


### 1b. Extract from a PDF (e.g. a resume)

Same extractor — just point it at a PDF. Define a schema for whatever you want to pull out.


In [ ]:
class WorkExperience(BaseModel):
    company: str = Field(examples=["Google"])
    title: str = Field(examples=["Software Engineer"])
    start_date: Optional[str] = Field(default=None, examples=["2021-01"])
    end_date: Optional[str] = Field(default=None, examples=["2024-06", "Present"])


class Resume(BaseModel):
    full_name: Optional[str] = Field(default=None, examples=["Sara Ali"])
    email: Optional[str] = Field(default=None, examples=["sara@example.com"])
    phone: Optional[str] = Field(default=None, examples=["+966 50 000 0000"])
    skills: List[str] = Field(default_factory=list, examples=[["Python", "SQL"]])
    experience: List[WorkExperience] = Field(default_factory=list)


In [ ]:
# Replace with a path to your own resume PDF, e.g. "/content/my_resume.pdf"
resume_pdf = "/content/my_resume.pdf"

# Uncomment when you have a real file:
# result = extractor.extract(source=resume_pdf, template=Resume)
# resume = Resume.model_validate(result.pages[0].extracted_data)
# print(resume)


### 1c. Tip — handling `.docx`

`DocumentExtractor` only accepts images and PDFs directly. For a Word document,
convert it to PDF first (e.g. with LibreOffice on Colab):

```bash
!apt-get -qq install -y libreoffice
!libreoffice --headless --convert-to pdf form.docx
```

then pass `form.pdf` to the extractor with the schema you want.


---
## Task 2: Arabic Document Processing with Qari-OCR

[Qari-OCR](https://huggingface.co/NAMAA-Space/Qari-OCR-v0.3-VL-2B-Instruct) is a VLM
fine-tuned from `Qwen2-VL-2B-Instruct` for Arabic documents. It preserves layout
(headers, tables) using HTML/Markdown and supports tashkeel (diacritics).


In [ ]:
%pip install -q transformers qwen-vl-utils accelerate "pdf2image" pillow
%pip install -q -U bitsandbytes
# pdf2image needs poppler on the system:
!apt-get -qq install -y poppler-utils


In [ ]:
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

MODEL_NAME = "NAMAA-Space/Qari-OCR-v0.3-VL-2B-Instruct"

print("Loading model... (first run downloads ~5GB)")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)
print("Model ready.")


In [ ]:
# Default prompt from the model card. The model returns plain text
# (with HTML/Markdown tags for layout when relevant).
DEFAULT_PROMPT = (
    "Below is the image of one page of a document, as well as some raw textual "
    "content that was previously extracted for it. Just return the plain text "
    "representation of this document as if you were reading it naturally. "
    "Do not hallucinate."
)


def ocr_image(image_path: str, prompt: str = DEFAULT_PROMPT, max_tokens: int = 2000) -> str:
    """Run Qari-OCR on a single image file and return the extracted text."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{image_path}"},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=max_tokens)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0]


### 2a. OCR a single Arabic image

In [ ]:
# Upload an Arabic image to Colab and put its path here
sample_image = "/content/some_email.jpg"

# Uncomment when you have a real file:
# text = ocr_image(sample_image)
# print(text)


### 2b. OCR a folder of Arabic PDFs

Each PDF page is rendered to an image, then sent to Qari-OCR. Results are saved as
`.md` files next to each PDF.


In [ ]:
import os
from pathlib import Path
from pdf2image import convert_from_path


def ocr_pdf(pdf_path: str, dpi: int = 200) -> str:
    """OCR every page of a PDF; return a single string with page separators."""
    pages = convert_from_path(pdf_path, dpi=dpi)
    chunks = []
    for i, page in enumerate(pages, start=1):
        tmp = f"/tmp/_page_{i}.png"
        page.save(tmp, "PNG")
        try:
            chunks.append(f"## Page {i}\n\n{ocr_image(tmp)}")
        finally:
            if os.path.exists(tmp):
                os.remove(tmp)
    return "\n\n".join(chunks)


def ocr_folder(folder: str, output_dir: str = None) -> dict:
    """OCR all PDFs in a folder. Saves a .md file per PDF and returns a dict {pdf: text}."""
    folder = Path(folder)
    output_dir = Path(output_dir) if output_dir else folder
    output_dir.mkdir(parents=True, exist_ok=True)

    results = {}
    pdfs = sorted(folder.glob("*.pdf"))
    print(f"Found {len(pdfs)} PDF(s) in {folder}")

    for pdf in pdfs:
        print(f"  → {pdf.name}")
        text = ocr_pdf(str(pdf))
        out = output_dir / f"{pdf.stem}.md"
        out.write_text(text, encoding="utf-8")
        results[pdf.name] = text

    print(f"Done. Outputs in {output_dir}")
    return results


In [ ]:
# Run on the example folder from the exercise:
folder_path = "data/HR/applications/ar/"

# Uncomment when the folder exists in your Colab session:
# results = ocr_folder(folder_path, output_dir="data/HR/applications/ar/_ocr")
# # Peek at the first result:
# first = next(iter(results))
# print(f"--- {first} ---")
# print(results[first][:1000])
